# Boardroom LLM Council (LangChain Version)

This version uses LangChain to orchestrate persona-driven debate and synthesis.
It is provider-agnostic via OpenAI-compatible endpoints.


## Install
If needed, install LangChain and the OpenAI client.


In [4]:
# If needed:
# !pip install -q langchain langchain-groq python-dotenv
!pip install dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]


## Provider Config (Groq)
Set environment variables for Groq.
- `GROQ_API_KEY`
- `GROQ_MODEL` (e.g., llama-3.3-70b-versatile)


In [5]:
from dotenv import load_dotenv
load_dotenv()


True

In [6]:
import os
from typing import List, Dict

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")
API_KEY = os.environ.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("Set GROQ_API_KEY in your environment.")

llm = ChatGroq(
    model=MODEL,
    api_key=API_KEY,
    temperature=0.4,
)

parser = StrOutputParser()


## Personas
Each persona has role, background, and focus.


In [7]:
PERSONAS = [
    {
        "name": "Regional Sales Head - Karnataka",
        "background": "Owns quarterly revenue in Karnataka; incentivized to grow local market share.",
        "style": "Pragmatic, numbers-driven, optimistic about local expansion.",
        "focus": "Local customer demand, sales trends, speed of market penetration."
    },
    {
        "name": "Country-wide Distribution Head",
        "background": "Responsible for nationwide logistics efficiency and cost control.",
        "style": "Risk-aware, process-oriented, skeptical of fragmented networks.",
        "focus": "Unit economics, logistics complexity, vendor SLAs, scaling."
    },
    {
        "name": "CEO",
        "background": "Balances growth with capital efficiency and long-term strategy.",
        "style": "Strategic, asks for trade-offs and long-term ROI.",
        "focus": "Strategic positioning, capital allocation, risk profile."
    }
]


## LangChain Orchestration
We build a prompt template for personas and run two rounds plus synthesis.


In [ ]:
def persona_prompt(p):
    return ChatPromptTemplate.from_messages([
        ("system",
         "You are {name}.\n"
         "Background: {background}\n"
         "Focus: {focus}\n"
         "Speaking style: {style}\n"
         "Be concise and practical. Provide 3-5 bullet points.\n"
         "If you make assumptions, label them.\n"),
        ("user", "{question}")
    ])# building the promt for each personality

def run_persona(p, question, temperature=0.4):
    chain = persona_prompt(p) | llm.bind(temperature=temperature) | parser
    return chain.invoke({
        "name": p["name"],
        "background": p["background"],
        "focus": p["focus"],
        "style": p["style"],
        "question": question,
    }) # runs the persona by building a prompt 

def run_council(question, personas=PERSONAS):
    # Round 1
    round1 = {}
    for p in personas:
        round1[p['name']] = run_persona(p, question, temperature=0.4)

    # Round 2
    round2 = {}
    for p in personas:
        other_points = "\n\n".join([f"{k}:\n{v}" for k, v in round1.items() if k != p['name']])
        rebuttal_q = (
            f"Question: {question}\n"
            f"Other viewpoints:\n{other_points}\n\n"
            "Respond with key rebuttals or alignments (3-5 bullets)."
        )
        round2[p['name']] = run_persona(p, rebuttal_q, temperature=0.5)

    # Synthesis by CEO (or last persona)
    ceo = [p for p in personas if 'CEO' in p['name']]
    ceo = ceo[0] if ceo else personas[-1]

    synthesis_input = "\n\n".join([
        f"{k} (Round 1):\n{v}" for k, v in round1.items()
    ] + [
        f"{k} (Round 2):\n{v}" for k, v in round2.items()
    ])

    synth_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are {name}.\n"
         "Synthesize a boardroom decision.\n"
         "Output format:\n"
         "Decision: <one sentence>\n"
         "Rationale: 3-5 bullets\n"
         "Risks: 2-3 bullets\n"
         "Next Steps: 3 bullets\n"),
        ("user", "{discussion}")
    ])

    synth_chain = synth_prompt | llm.bind(temperature=0.3) | parser
    final = synth_chain.invoke({
        "name": ceo["name"],
        "discussion": synthesis_input,
    })

    return round1, round2, final


## Example Run


In [9]:
question = "Should Acme Corporation set up its own distribution network in Karnataka or outsource it?"

round1, round2, final = run_council(question)

print("=== Round 1 ===")
for k, v in round1.items():
    print(f"\n[{k}]\n{v}")

print("\n=== Round 2 ===")
for k, v in round2.items():
    print(f"\n[{k}]\n{v}")

print("\n=== Final Synthesis ===\n")
print(final)


=== Round 1 ===

[Regional Sales Head - Karnataka]
To determine whether Acme Corporation should set up its own distribution network in Karnataka or outsource it, let's consider the following points:

* **Control and Quality**: Setting up our own distribution network would give us greater control over the delivery process, allowing us to ensure high-quality service and faster issue resolution. (Assumption: Quality of service is a key differentiator for Acme Corporation)
* **Cost and Investment**: Outsourcing distribution would require lower upfront investment, but may lead to higher long-term costs due to potential markups by the outsourcing partner. In contrast, setting up our own network would require significant initial investment, but could lead to cost savings in the long run.
* **Speed of Market Penetration**: Outsourcing distribution could allow us to quickly penetrate the Karnataka market, as we could leverage the existing network and expertise of the outsourcing partner. (Assum